# 预训练实验（Mac 本地缩小型）

**结论先行**：教程的 Llama-3-8B（8×NPU 训数天）在本地 Mac 不可行；本实验改为**从零预训练一个微型 Llama 架构模型**（91.4M 参数），数据集（bookcorpus100mb 已分词）、分词器（Llama-3.2 词表 128256）、训练流程与教程完全一致。

**实测环境**：M5 / 34GB 统一内存 / torch 2.11 + MPS。已冒烟验证 10 步训练正常收敛（初始 loss 11.83 ≈ ln(128256)=11.76 理论值）。

| 预算 | max_steps | 预计耗时（fp16, eff batch 8） |
|---|---|---|
| 快速验证 | 250 | ~1.4 小时 |
| 半数据 | 756 | ~4 小时 |
| 完整 1 epoch（1240 万 token） | 1513 | ~8 小时 |

In [8]:
# 数据与分词器路径（绝对路径，避免工作目录歧义）
TOKENIZER_PATH = "/Users/wjx/Desktop/知识库/LLM/动手学习大模型/models/Llama-3.2-1B"
CHUNKED_DS_PATH = "/Users/wjx/Desktop/知识库/LLM/动手学习大模型/datasets/bookcorpus/chunked_ds"
OUTPUT_DIR = "/Users/wjx/Desktop/知识库/LLM/动手学习大模型/models/bookcorpus-pretrain-91m"

from datasets import load_from_disk
from transformers import AutoTokenizer
from transformers import LlamaConfig, LlamaForCausalLM

# 加载已分块数据集：train 12098 条 × 1024 token，test 16 条
ds = load_from_disk(CHUNKED_DS_PATH)

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
tokenizer.pad_token = tokenizer.eos_token  # 预训练自回归：pad = eos = 128001

In [6]:
# 从零构建微型 Llama 模型（随机初始化，无预训练权重）
config = LlamaConfig(
    vocab_size=128256,               # 与 Llama-3.2 分词器一致（必须匹配）
    hidden_size=512,
    intermediate_size=1408,
    num_hidden_layers=8,
    num_attention_heads=8,
    num_key_value_heads=8,
    max_position_embeddings=1024,    # 与 chunk 长度一致
    tie_word_embeddings=True,        # 词表层与输出层共享，省 ~65M 参数
)
model = LlamaForCausalLM(config)
n_params = sum(p.numel() for p in model.parameters())
print(f"参数量: {n_params/1e6:.1f}M")

# 想训更大：hidden_size=768, num_hidden_layers=12, intermediate_size=3072
# -> ~211M 参数，速度约减半，可按需替换

参数量: 91.4M


In [10]:
# 训练参数（对标教程 Llama-3-8B 配置，缩放到本地可跑）
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

MAX_STEPS = 250  # 快速验证；完整 1 epoch 改 1513（~8h），半数据 756（~4h）

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # 有效 batch = 2×4 = 8
    learning_rate=3e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    max_steps=MAX_STEPS,
    eval_strategy="steps",
    eval_steps=200,
    per_device_eval_batch_size=4,
    logging_strategy="steps",
    logging_steps=25,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    seed=42,
    fp16=True,                       # MPS fp16 实测比 fp32 快 ~2×；若报错改 fp16=False
    report_to=[],                    # 不上传 wandb；本地可看日志
)

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)  # 因果 LM：labels 自动右移

trainer = Trainer(
    model=model,
    args=args,
    processing_class=tokenizer,   # transformers>=4.57 新参数名（旧名 tokenizer= 已弃用）
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    data_collator=collator,
)

In [11]:
# 开始训练（中断后可加 resume_from_checkpoint=True 续训）
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128001, 'bos_token_id': 128000, 'pad_token_id': 128001}.
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
200,5.043900,5.042199


TrainOutput(global_step=250, training_loss=5.842304595947265, metrics={'train_runtime': 5878.205, 'train_samples_per_second': 0.34, 'train_steps_per_second': 0.043, 'total_flos': 315787051008000.0, 'train_loss': 5.842304595947265, 'epoch': 0.1653165812530997})

In [12]:
# 保存最终模型（含分词器与配置）
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("已保存到:", OUTPUT_DIR)

已保存到: /Users/wjx/Desktop/知识库/LLM/动手学习大模型/models/bookcorpus-pretrain-91m


In [19]:
# 验证：加载训练产物并生成一小段（91M 模型 + 12M token，输出会是破碎的小写英文，属正常现象）
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

m = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR)
t = AutoTokenizer.from_pretrained(OUTPUT_DIR)
t.pad_token = t.eos_token

input_ids = t.encode("she killed the man", return_tensors="pt")
with torch.no_grad():
    out = m.generate(input_ids, max_new_tokens=50, do_sample=True, temperature=0.8)

text = t.decode(out[0])
text = text.replace(t.bos_token, "")   # 去掉输出中的 <|begin_of_text|> 特殊符号
print(text.strip())

she killed the manshe had you want it.`` i don't want you're going to be a last.`` you want the other hands.`` i 'll be, '' '' he said, but he was not to him.


# 实测数据与说明

- **速度**（MPS 实测）：fp16 + eff batch 8 = 19.1s/step（~430 token/s）；fp32 ≈ 40s/step。
- **Loss 轨迹参考**：随机初始化 11.83 → 12 步后 9.87；跑 250 步预计到 9 以下。
- **为什么不在 Mac 上训 8B**：教程配置需 8×NPU 显存 + 数天机时；本地 34GB 内存跑 8B 连梯度都放不下。
- **提速选项**：`per_device_train_batch_size=4`（可能提升 GPU 利用率）；进一步缩小 hidden_size=384 / 6 层（~50M 参数，再快 ~1.6×）。
- **预期产物**：一个能"鹦鹉学舌"小说文风的 91M 模型，用于理解预训练全流程；真实能力需 8B 级别 + 数 T token。